# Langchain quickstart

In this notebook you will see how easy it is to create your first chain usiung EPAM Dial as a LLM proxy. In just few steps we will use EPAM Dial to send messages to LLM. In next step we will introduce output parser to show how to use chains.
Notebook based on https://python.langchain.com/v0.1/docs/get_started/quickstart/ adapted to EPAM Dial.

To create LLM instance (in this case EPAM Dial) use:

In [1]:
import os
from langchain_openai import AzureChatOpenAI

epam_dial = AzureChatOpenAI(
        api_key         = os.environ['OPENAI_API_KEY'],
        api_version     = "2023-07-01-preview",
        azure_endpoint  = "https://ai-proxy.lab.epam.com",
        model           = "gpt-4o-mini-2024-07-18",
        temperature     = 0.0
    )

Once you've initialized the LLM of your choice, we can try using it! Let's ask it what LangSmith is - this is something that wasn't present in the training data so it shouldn't have a very good response.

In [3]:
epam_dial.invoke("Tell me a joke about software developers")

AIMessage(content='Why do software developers prefer dark mode?\n\nBecause light attracts bugs!', response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 14, 'total_tokens': 27}, 'model_name': 'gpt-35-turbo', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None, 'content_filter_results': {}}, id='run-8786f024-1461-4a14-88d2-c9d5b7c6e5da-0', usage_metadata={'input_tokens': 14, 'output_tokens': 13, 'total_tokens': 27})

We can also guide its response with a prompt template. Prompt templates convert raw user input to better input to the LLM.

In [4]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a world class comedian that tells the longest jokes ever."),
    ("user", "{input}")
])

We can now combine these into a simple LLM chain:

In [5]:
chain = prompt | epam_dial 

We can now invoke it and ask the same request. It should respond in a more longer form!

In [6]:
chain.invoke({"input": "Tell me a joke about software developers"})

AIMessage(content="Alright, buckle up because this is going to be a long one!\n\nOnce upon a time, in a land far, far away, there was a kingdom ruled by a wise and tech-savvy king. This king had a grand vision for his kingdom - to create the most advanced and efficient software ever known to mankind. To achieve this, he summoned the most brilliant software developers from all corners of the world to work on his ambitious project.\n\nNow, among these developers, there were two individuals named Alex and Ben. Alex was a seasoned developer with years of experience, while Ben was a fresh graduate eager to prove himself. The king, impressed by their skills, assigned them a crucial task - to develop a program that could accurately predict the weather for the entire kingdom.\n\nExcited about the opportunity, Alex and Ben set off on their journey. They spent countless hours brainstorming, coding, and testing their program. Days turned into weeks, and weeks turned into months. They faced numero

The output of a ChatModel (and therefore, of this chain) is a message. However, it's often much more convenient to work with strings. Let's add a simple output parser to convert the chat message to a string. This is great example of how to use chains.

In [7]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

We can now add this to the previous chain:

In [8]:
chain = prompt | epam_dial | output_parser

We can now invoke it and ask similar question. The answer will now be a string (rather than a ChatMessage).

In [10]:
chain.invoke({"input": "Tell me a joke about software bootcamps"})

'Alright, buckle up because this joke is going to take you on a wild ride through the world of software bootcamps!\n\nSo, there was once a guy named Tim who decided to enroll in a software bootcamp. He was super excited to learn all about coding and become a software developer. Little did he know, this bootcamp was notorious for its incredibly long and intense program.\n\nTim arrived on the first day, ready to dive headfirst into the world of coding. The instructor, let\'s call him Professor Byte, welcomed everyone and said, "Alright, folks, we\'re going to start with the basics. Today, we\'re going to learn about variables."\n\nTim thought, "Great! Variables, I can handle that." But little did he know, Professor Byte had a unique teaching style. He started explaining variables, and Tim quickly realized that this was going to be a long journey.\n\nDays turned into weeks, and weeks turned into months. Tim and his fellow bootcampers were knee-deep in coding exercises, debugging sessions,